# Demo codes of Quantum Computing for Nuclear Physics

- (1) Hamiltonian mapping
    - a. Jordan-Wigner mapping of nuclear Hamiltonians
    - b. Pairing Hamiltonian
    - c. comparison with Chemical Hamiltonians
- (2) VQE with UCC ansatz
- (3) Algorithms with Time-evolution
    - a. Time-evolution with Trotterization
    - b. resource estimation for (E-)FTQC methods
        - QPE
        - Qkrylov
        - ODMD
    - c. QSCI/SQD demo for small systems
- (4) ODMD examples for two-neutron systems

## Pre-requisites



In [1]:
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2
from qiskit_algorithms.optimizers import SLSQP, NFT, L_BFGS_B, ADAM
from qiskit_algorithms.minimum_eigensolvers import AdaptVQE, VQE
from qiskit_ibm_runtime.fake_provider import FakeTorino
from qiskit_nature.second_q.circuit.library.ansatzes import UCC
from qiskit.primitives import Estimator
import seaborn as sns
cols_hist = sns.color_palette("deep")


## To run the demo codes: virtual environment

After a certain version for Python and pip (around 3.11),
the Python community proposed [PEP 668](https://peps.python.org/pep-0668/) for managing python packages.
Users are now strongly recommended to use virtual environments to avoid conflicts between packages and conflicts with the system itself.

You can create a virtual environment with the following command:

```bash
$python3 -m venv name_of_your_env
```

Then, you can activate the virtual environment with the following command:

```bash
$source path_to_your_env/bin/activate
```

After activating the virtual environment, you can install the required packages with the following command:

```bash
$pip install package_name
```

or 

```bash
$pip install -r requirements.txt
```

if `requirements.txt` is provided in the repository.


In the following, we assume you already have cloned `nuqulib` module and using `pip`.

```bash
source path_to_your_venv/bin/activate
pip install -r /path_to_nuqulib/requirements.txt
pip install -e /path_to_NuQuLib
```

In [2]:
from nuqulib import *

int_file_path = "tests/interaction_file/"
int_file_path = "interaction_file/"


## (1) Hamiltonian mapping

### (1)-a: Jordan-Wigner mapping of nuclear Hamiltonians

In the following, we demonstrate the usage of the `NuQuLib` package by taking smaller systems with only a few (valence) nucleons.

First, we read `ckpot` effective interaction for 0p shell (p1/2 and p3/2 orbitals on top of 4He core) and map it to the qubit Hamiltonian using Jordan-Wigner transformation.

NuQuLib allows to read the interaction files for NN and 3N interactions in e.g. `snt` (`kshell.snt` in NuHamil code) format.
One can check the mapping is correct by comparing the energy of the ground state of fully occupied systems in the specified model space.

In the following, we will see both valence (p, sd, pf) and NCSM (emax=0, 1) cases give correct results.

#### valence space

We will consider the following fully occupied systems in the valence shell model spaces:

1. 16O in p shell
2. 40Ca in sd shell


In [ ]:
fns = ["ckpot.snt", "usdb.snt"]
ZNs = [ (8, 8), (20, 20) ]

for idx in range(len(fns)):
    filename_snt = int_file_path + fns[idx]
    print("filename_snt:", filename_snt)
    Z, N = ZNs[idx]
    hamil = Hamiltonian(filename_snt, Z, N)

    mapping_method = "JordanWigner"
    Hdict_M = hamil.get_mscheme_H(opform=True)
    H_1b, H_pp, H_nn, H_pn = hamil.mapping_opform(Hdict_M, mapping_method)

    n_qubits = hamil.n_qubits
    qc = QuantumCircuit(n_qubits)
    qc.x(range(n_qubits))  # prepare all qubits in |1> state

    # measurement of Hamiltonian
    estimator = StatevectorEstimator()
    job = estimator.run([(qc, H_1b), (qc, H_pp), (qc, H_nn), (qc, H_pn)])
    results = job.result()
    E_1b, E_pp, E_nn, E_pn = [ results[i].data.evs for i in range(len(results))]
    E_total = E_1b + E_pp + E_nn + E_pn 
    print("Etot", E_total, "E_1b: ", E_1b, "<pp>", E_pp, "<nn>", E_nn, "<pn>", E_pn)


#### NCSM (No-core shell model) space

NN interaction is the so-called EM500 with SRG (1.8 fm^{-1} cutoff) and HO basis with hw=20 MeV and emax=1.
The 3N interaction called local-nonlocal (lnl) interaction is used.

Detailed parameters are ...

One should keep in mind that the results will change depending on the choice of the interaction and the settings to generate the Hamiltonian.
Within these settings, the ground state energy of ${}^{16}\text{O}$ would be -131.83565 MeV.

In [ ]:
emax = 1
filename_snt = int_file_path + "TwBME-HO_NN-only_N3LO_EM500_srg1.8_hw20_emax1_e2max2.kshell.snt"
fn_3NF = int_file_path + "ThBME_lnl_ms1_2_1.readable.txt"
Z = proton_number  = N = neutron_number = 8

hamil = Hamiltonian(filename_snt, Z, N, ncsm=True, verbose=False, emax_truncate=emax,
                    e3max=emax, fn_3NF=fn_3NF)
mapping_method = "JordanWigner"
Hdict_M = hamil.get_mscheme_H(opform=True)
H_1b, H_pp, H_nn, H_pn = hamil.mapping_opform(Hdict_M, mapping_method)

Hamil_NCSM_NN = H_1b + H_pp + H_nn + H_pn
hamil.set_mscheme_3NF()
H_3b = hamil.mapping_3NF_Mscheme()
Hamil_NCSM_NN3NF = Hamil_NCSM_NN + H_3b

n_qubits = hamil.n_qubits
qc = QuantumCircuit(n_qubits)
qc.x(range(n_qubits))  # prepare all qubits in |1> state

# measurement of Hamiltonian
estimator = StatevectorEstimator()
job = estimator.run([(qc, H_1b), (qc, H_pp), (qc, H_nn), (qc, H_pn), (qc, H_3b)])
results = job.result()
E_1b, E_pp, E_nn, E_pn, E_3n = [ results[i].data.evs for i in range(len(results))]
E_total = E_1b + E_pp + E_nn + E_pn + E_3n
print("Etot", E_total, "E(NN)", E_1b + E_pp + E_nn + E_pn)
print("E_1b: ", E_1b, "<pp>", E_pp, "<nn>", E_nn, "<pn>", E_pn, "<3b>", E_3n)

### (1)-b: Pairing Hamiltonian (JW)

Pairing Hamiltonian is a simplified model of the nuclear interaction or a simplified model of BCS pairing. The Hamiltonian with a global pairing interaction is given by

$$
H = \sum_{i} \epsilon_i a_i^{\dagger} a_i - \frac{g}{4} \sum{i,j} a_i^{\dagger} a_{\bar{i}}^{\dagger} a_{\bar{j}} a_j
$$

where $a_i^{\dagger}$ and $a_i$ are the creation and annihilation operators for the nucleon in the $i$-th orbital, $\epsilon_i$ is the single-particle energy of the $i$-th orbital, $g$ is the pairing strength, and $\bar{i}$ is the time-reversed orbital of $i$.

In [ ]:
Norb = 12
Nocc = 6
gval = 0.33
hamil = PairingHamiltonian(Norb, Nocc, gval)
Hamil_Pairing = hamil.encoding()

# Make a time evolution operator U = exp(-iHt)
delta_t = 0.1
Unitary_P = PauliEvolutionGate(Hamil_Pairing, delta_t, synthesis=SuzukiTrotter(order=1,reps=1))
print("U:", Unitary_P)

# Make a quantum circuit (1 ancilla + target qubits)
qc = QuantumCircuit(Norb+1)
Unitary_P = Unitary_P.control(1)
qc.append(Unitary_P, range(Norb+1))
qc = qc.decompose(reps=5)
print("# of gates, raw circuit:", qc.count_ops())

### (1)-c: Chemical Hamiltonians 

According to the Pennylane documentation [here](https://docs.pennylane.ai/en/stable/introduction/chemistry.html), `qchem` module provides molecular Hamiltonians.

One can specify the molecule by its symbols and geometry, and then the Hamiltonian can be generated using the `qml.qchem.molecular_hamiltonian` function.

Similarly, Qiskit provides molecular Hamiltonians via e.g. `qiskit_nature` module.

In [ ]:
symbols = ["Li", "H"]
geometry = np.array([[0.0, 0.0, 0.0], [0.0, 0.0, 2.969280527]])
molecule = qml.qchem.Molecule(symbols, geometry)
H, qubits = qml.qchem.molecular_hamiltonian(
    molecule,
    active_electrons=2,
    active_orbitals=5,
)
def circuit(params):
    qml.ApproxTimeEvolution(H, time=0.1, n=1)
    return [qml.expval(qml.PauliZ(i)) for i in range(qubits)]

dev = qml.device("default.qubit", wires=qubits)
params = np.random.rand(qubits)
compiled_qnode = qml.compile(
    qml.QNode(circuit, device=dev),
    basis_set=["CNOT", "RX", "RY", "RZ"])

spec = qml.specs(compiled_qnode)(params)

print("qubits:", qubits, "Gates:", dict(spec["resources"].gate_types))

## (2): VQE with UCC ansatz



The key components of the VQE algorithm is to prepare a trial wavefunction, given as parametrized quantum circuit, and to minimize the expectation value of the Hamiltonian with respect to this wavefunction.

We adopt the unitary coupled cluster (UCC) ansatz here.

To start rather arbitrarily, let us define a function to generate excitation operators on top of a naive filling configuration. This will be referred in the UCC class, so it should be defined in this manner.

In [ ]:
def custom_excitation_list(num_spatial_orbitals, num_particles):
    global hole_p, particle_p, hole_n, particle_n
    my_custom_excitations = [ ]
    ## Single excitations
    for h_p in hole_p:
        for p_p in particle_p:
            my_custom_excitations.append( ((h_p,), (p_p,)) )
    for h_n in hole_n:
        for p_n in particle_n:
            my_custom_excitations.append( ((h_n,), (p_n,)) )

    ## pp doubles
    for h_p_1 in hole_p:
        for h_p_2 in hole_p:
            if h_p_1 == h_p_2: continue
            for p_p_1 in particle_p:
                for p_p_2 in particle_p:
                    if p_p_1 == p_p_2: continue
                    my_custom_excitations.append( ((h_p_1, h_p_2), (p_p_1, p_p_2)) )
    ## nn doubles
    for h_n_1 in hole_n:
        for h_n_2 in hole_n:
            if h_n_1 == h_n_2: continue
            for p_n_1 in particle_n:
                for p_n_2 in particle_n:
                    if p_n_1 == p_n_2: continue
                    my_custom_excitations.append( ((h_n_1, h_n_2), (p_n_1, p_n_2)) )
    ## pn doubles
    for h_p in hole_p:
        for h_n in hole_n:
            for p_p in particle_p:
                for p_n in particle_n:
                    my_custom_excitations.append( ((p_p, p_n), (h_p, h_n)) )
    return my_custom_excitations

### 6He (2n on 0p shell)

Egs (J=0) =  -3.90981 MeV

In [ ]:
np.random.seed(42)  
fn = int_file_path+"ckpot.snt"
Z = 0; N = 2
Hamil_Shellmodel, H_JW, proton_qubits, neutron_qubits = get_Hamiltonian(fn, Z, N)
n_qubits = Hamil_Shellmodel.n_qubits
mapper = JordanWignerMapper()

# Define the initial state&UCC ansatz
init_state = QuantumCircuit(n_qubits)
init_state.x(9); init_state.x(10); hole_p = [ ]; hole_n = [9, 10]

particle_p = [q for q in proton_qubits if q not in hole_p]
particle_n = [q for q in neutron_qubits if q not in hole_n]

display(init_state.draw())

vqe_ansatz = UCC(
    n_qubits//2,
    num_particles=(Z, N),
    initial_state=init_state,
    qubit_mapper = mapper,
    preserve_spin=True,
    reps=1,
    excitations = custom_excitation_list
)

optimizer = SLSQP(maxiter=200)
print("Number of parameters: ", vqe_ansatz.num_parameters)
print("vqe.excitations: ", vqe_ansatz._excitation_list)
print("Gate counts:", vqe_ansatz.decompose(reps=5).count_ops())

def store_intermediate_result(eval_count, parameters, mean, std):
    counts.append(eval_count)
    values.append(mean)

counts = []
values = []
vqe = VQE(Estimator(), vqe_ansatz, optimizer, callback=store_intermediate_result)
vqe.initial_point = np.random.normal(0, 0.1, vqe_ansatz.num_parameters)
vqe_result = vqe.compute_minimum_eigenvalue(H_JW)
binding_energy = vqe_result.optimal_value
print("E(VQE) = ", binding_energy)

In [ ]:
counts_vqe = list(counts)
values_vqe = list(values)
Eexact =  -3.90981
fig = plt.figure(figsize=(5, 3))
plt.plot(values_vqe, color=cols_hist[0], label='VQE', zorder=1)
plt.axhline(y=Eexact, color=cols_hist[3], linestyle='--', label='Exact', zorder=0)
plt.xlabel('Evaluation Count')
plt.ylabel('Energy (MeV) of $^6$He')
plt.legend()
plt.rcParams["font.size"] = 14
plt.savefig("figure/vqe_energy_6He.pdf", bbox_inches='tight', pad_inches=0.01)
plt.show()

### 6Li (p-n on 0p shell)

Egs = -5.43299 MeV (J=1)

In [ ]:
np.random.seed(42)  

fn = int_file_path + "ckpot.snt"
Z = N = 1
Hamil_Shellmodel, H_JW, proton_qubits, neutron_qubits = get_Hamiltonian(fn, Z, N)
n_qubits = Hamil_Shellmodel.n_qubits
mapper = JordanWignerMapper()

# Define the initial state&UCC ansatz
init_state = QuantumCircuit(n_qubits)
init_state.x(3); init_state.x(10); hole_p = [3]; hole_n = [10]

particle_p = [q for q in proton_qubits if q not in hole_p]
particle_n = [q for q in neutron_qubits if q not in hole_n]

display(init_state.draw())

vqe_ansatz = UCC(
    n_qubits//2,
    num_particles=(Z, N),
    initial_state=init_state,
    qubit_mapper = mapper,
    preserve_spin=True,
    reps=1,
    excitations = custom_excitation_list
)

optimizer = SLSQP(maxiter=200)
print("Number of parameters: ", vqe_ansatz.num_parameters)


def store_intermediate_result(eval_count, parameters, mean, std):
    counts.append(eval_count)
    values.append(mean)

counts = []
values = []
vqe = VQE(Estimator(), vqe_ansatz, optimizer, callback=store_intermediate_result)
vqe.initial_point = np.random.normal(0, 0.1, vqe_ansatz.num_parameters)
vqe_result = vqe.compute_minimum_eigenvalue(H_JW)
binding_energy = vqe_result.optimal_value
#print("Parameters: ", vqe_result.optimal_point)

print("E(VQE) = ", binding_energy)
print("Gate counts:", vqe_ansatz.decompose(reps=5).count_ops())

In [ ]:
counts_vqe = list(counts)
values_vqe = list(values)
Egs = -5.43299
Eex1 = -5.008799945408569
fig = plt.figure(figsize=(5, 3))
plt.plot(values_vqe, color=cols_hist[0], label='VQE', zorder=1)
plt.axhline(y=Egs, color=cols_hist[3], linestyle='--', label='Exact J=1', zorder=0)
plt.axhline(y=Eex1, color=cols_hist[2], linestyle='--', label='Exact J=3', zorder=0)
plt.xlabel('Evaluation Count')
plt.ylabel('Energy (MeV) of $^6$Li')
plt.legend()
plt.rcParams["font.size"] = 14
plt.savefig("figure/vqe_energy_6Li.pdf", bbox_inches='tight', pad_inches=0.01)
plt.show()

While 6He (2n in 0p shell) is reproduced correctly, 6Li (p-n) is not.

Since both proton and neutron degrees of freedom are involved, the circuit becomes a bit complicated and deeper than the 6He case.
Even in such a small system, one can easily get stuck in local minima, E= -5.00880 (J=3) state, instead of the true g.s. -5.43299 (J=1).

Another possibility to reduce the circuit depth is to use the adaptive VQE (Adapt-VQE) algorithm, which adaptively selects the excitation operators from the pool of all possible excitations based on the gradient of the energy.
In that case, one should also be careful about the initialization of the trial wavefunction, optimization method, etc.

See also [O.Kiss et al.: Quantum computing of the 6Li nucleus via ordered unitary coupled clusters, Phys. Rev. C 106, 034325 (2022)]( https://doi.org/10.1103/PhysRevC.106.034325).

## (3) Algorithms with Time-evolution

### (3)-a: Time-evolution with Trotterization

Time-evolution is a key component of many quantum algorithms.
By means of time-evolution, constructed via Trotterization, one can implement various quantum algorithms such as quantum phase estimation (QPE), quantum Krylov (Qkrylov), and observable dynamic mode decomposition (ODMD).

In the following, only the first order and single step Suzuki-Trotter decomposition is used:

$$
\hat{U}(t) = e^{-i \hat{H} t} \approx e^{-i \hat{H}_1 t} e^{-i \hat{H}_2 t} \cdots e^{-i \hat{H}_n t}
$$

where $\hat{H}_i$ are the terms in the Hamiltonian given as e.g. Qiskit's `SparsePauliOp` object containing the Pauli strings and their (real) coefficients.


Taking NCSM Hamiltonian as an example, we show a sample code to perform time-evolution with Trotterization.


In [ ]:
fn_2NF = int_file_path+"TwBME-HO_NN-only_N3LO_EM500_srg1.8_hw20_emax1_e2max2.kshell.snt"
fn_3NF = int_file_path+"ThBME_lnl_ms1_2_1.readable.txt"
Z = N = 3 

hamil = Hamiltonian(fn_2NF, Z, N, ncsm=True, verbose=False, emax_truncate=emax, e3max=emax, fn_3NF=fn_3NF)
mapping_method = "JordanWigner"
Hdict_M = hamil.get_mscheme_H(opform=True)
H_1b, H_pp, H_nn, H_pn = hamil.mapping_opform(Hdict_M, mapping_method)
Hamil_NN = H_1b + H_pp + H_nn + H_pn
hamil.set_mscheme_3NF()
H_3b = hamil.mapping_3NF_Mscheme()
Hamil_NN3NF = Hamil_NN + H_3b

dt = 0.1
trotter_steps = 1
U_NN = PauliEvolutionGate(Hamil_NN, dt, synthesis=SuzukiTrotter(order=1,reps=trotter_steps))
U_3N = PauliEvolutionGate(Hamil_NN3NF, dt, synthesis=SuzukiTrotter(order=1,reps=trotter_steps))

qc = QuantumCircuit(hamil.n_qubits)
qc.append(U_NN, range(hamil.n_qubits))
qc = qc.decompose(reps=5)
print("# of gates (NN-only):", qc.count_ops())

qc = QuantumCircuit(hamil.n_qubits)
qc.append(U_3N, range(hamil.n_qubits))
qc = qc.decompose(reps=5)
print("# of gates (NN+3NF): ", qc.count_ops())

### (3)-b: resource estimation for (E-)FTQC methods

#### Quantum Phase Estimation (QPE)

QPE is the most fundamental quantum algorithm to estimate the eigenvalues of a Hamiltonian, i.e. interacting quantum many-body system.

The basic idea is to prepare register qubits, called ancilla qubits, to extract the eigenvalues of the Hamiltonian by means of controlled time-evolution.
The number of ancilla qubits determines the precision of the eigenvalue estimation, i.e. the number of bits in the binary representation of the eigenvalue.
If one adds one ancilla qubit, the precision is doubled, i.e. one can estimate the eigenvalue up to $2^{-1}$.
However, adding one ancilla qubit requires an additional controlled time-evolution gate and thus increases the circuit depth.

Under the given number of ancilla qubits, $N_a$, the required time-evolution operator is given as

$$
\sum_{k=0}^{N_a-1} \hat{U}^{2^k}
$$

In the case of $N_a=10$, the number of controlled $U$ operation becomes 1024-1=1023.
This scaling prohibits the direct implementation of QPE for large systems in current quantum computers.


In [ ]:
def resource_estimation_QPE(N_a, N_q, H_mapped, dt, trotter_steps=1):
    Unitary = PauliEvolutionGate(H_mapped, dt, synthesis=SuzukiTrotter(order=1,reps=trotter_steps))
    # We don't construct the full circuit here, just construct a single controlled unitary    
    qc = QuantumCircuit(1+N_q)
    cU = Unitary.control(1)
    qc.append(cU, [0] + list(range(1, 1+N_q)))
    qc = qc.decompose(reps=5)
    cU_ops = qc.count_ops()
    print("# of gates c-U:", cU_ops)
    dict_ops = { }
    for ith in range(N_a):
        for k, v in cU_ops.items():
            if k not in dict_ops:
                dict_ops[k] = 0
            dict_ops[k] += v * 2**ith
    print("# of gates for QPE:", dict_ops) 
    print("# of CNOTs:", str("%.1e" % (dict_ops.get('cx', 0) )))

Na= 10 # number of ancilla

print("valence p shell")
hamil, H_mapped, proton_qubits, neutron_qubits = get_Hamiltonian(int_file_path+"ckpot.snt", 2, 2)
Nq = hamil.n_qubits
resource_estimation_QPE(Na, Nq, H_mapped, dt=0.1, trotter_steps=1)

print("NCSM NN+3NF")
fn_2NF = int_file_path+"TwBME-HO_NN-only_N3LO_EM500_srg1.8_hw20_emax1_e2max2.kshell.snt"
fn_3NF = int_file_path+"ThBME_lnl_ms1_2_1.readable.txt"
hamil, H_mapped, proton_qubits, neutron_qubits = get_Hamiltonian(fn_2NF, Z, N, fn_3NF=fn_3NF)
Nq = hamil.n_qubits
resource_estimation_QPE(Na, Nq, H_mapped, dt=0.1, trotter_steps=1)

#### Quantum Krylov (Qkrylov)


Quantum Krylov (QKrylov) here refers to the quantum algorithm to estimate the eigenvalues of a Hamiltonian by means of controlled time-evolution.
Time evolution operators are applied to the initial state to generate non-orthogonal states,

$$
\ket{\Phi}_0, e^{-iHt_1}\ket{\Phi}_0, \ldots, e^{-iHt_{M}}\ket{\Phi}_0,
$$

which are then used to construct the Krylov subspace.
One can then estimate the eigenvalues of the Hamiltonian by means of a method to solve the generalized eigenvalue problem (GEVP) in the Krylov subspace.

$$
\begin{align}
\tilde{H} \ket{\Phi} & = E N \ket{\Phi}\\
N_{kl} & = \langle \Phi_k \ket{\Phi_l} = \bra{\Phi_0} e^{-i(t_l-t_k)H} \ket{\Phi_0} \\
\tilde{H}_{kl} & = \bra{\Phi_k} H \ket{\Phi_l} = \bra{\Phi_0} H e^{-i(t_l-t_k)H} \ket{\Phi_0}
\end{align}
$$

To measure the overlap, the following circuits are used:

![](https://sotayoshida.github.io/Lecture_SummerSchool2025/_images/overlap_Re.png)

![](https://sotayoshida.github.io/Lecture_SummerSchool2025/_images/overlap_Im.png)

The above/below circuits are used to measure the real/imaginary part of the overlap, respectively.

For only one component of the overlap, we need four times controlled time-evolution.
Since the diagonal elements of the overlap can be evaluated by hand, off-diagonal elements are to be measured.
If one considers $N_\mathrm{iter.}$ iterations, the total number of off-diagonal elements to be measured is $N_\mathrm{iter.}(N_\mathrm{iter.}-1)/2$.

Considering the Hamiltonian counter part, the total number of controlled time-evolution gates is to be multiplied by the number of Hamiltonian terms, $N_{\hat{H}}$.

Finally, the total number of controlled time-evolution gates is given as

$$
2  N_\mathrm{iter.} (N_\mathrm{iter.}-1) ( 1 + N_{\hat{H}}) \simeq 2 N_\mathrm{iter.}^2 N_{\hat{H}}
$$

In [ ]:
def resource_estimation_QKrylov(Niter, H_mapped, dt=1.0, trotter_steps=1):
    N_H = len(H_mapped.paulis)
    Unitary = PauliEvolutionGate(H_mapped, dt, synthesis=SuzukiTrotter(order=1,reps=trotter_steps))
    # We don't construct the full circuit here, just construct a single controlled unitary    
    qc = QuantumCircuit(1+H_mapped.num_qubits)
    cU = Unitary.control(1)
    qc.append(cU, [0] + list(range(1, 1+H_mapped.num_qubits)))
    qc = qc.decompose(reps=5)
    cU_ops = qc.count_ops()
    print("# of gates c-U:", cU_ops)
    dict_ops = { }
    for k, v in cU_ops.items():
        dict_ops[k] = v * ( 2 * Niter * (Niter -1) * (1+N_H))
    print("# of gates for QKrylov:", dict_ops)
    print("# of CNOTs:", str("%.1e" % (dict_ops.get('cx', 0) )))

hamil, H_mapped, proton_qubits, neutron_qubits = get_Hamiltonian(int_file_path+"ckpot.snt", 2, 2)
resource_estimation_QKrylov(Niter=50, H_mapped=H_mapped)

#### Observable Dynamic Mode Decomposition (ODMD)



In [ ]:
def resource_estimation_ODMD(Niter, H_mapped, dt=1.0, trotter_steps=1):
    N_H = len(H_mapped.paulis)
    Unitary = PauliEvolutionGate(H_mapped, dt, synthesis=SuzukiTrotter(order=1,reps=trotter_steps))
    # We don't construct the full circuit here, just construct a single controlled unitary    
    qc = QuantumCircuit(1+H_mapped.num_qubits)
    cU = Unitary.control(1)
    qc.append(cU, [0] + list(range(1, 1+H_mapped.num_qubits)))
    qc = qc.decompose(reps=5)
    cU_ops = qc.count_ops()
    print("# of gates c-U:", cU_ops)
    dict_ops = { }
    for k, v in cU_ops.items():
        dict_ops[k] = v * ( 2 * Niter * (Niter -1))
    print("# of gates for ODMD:", dict_ops)
    print("# of CNOTs:", str("%.1e" % (dict_ops.get('cx', 0) )))

hamil, H_mapped, proton_qubits, neutron_qubits = get_Hamiltonian(int_file_path+"ckpot.snt", 2, 2)
resource_estimation_ODMD(Niter=50, H_mapped=H_mapped)

### (3)-c: QSCI/SQD demo for small systems

As a simple example, we will consider a variant of the Quantum Selected Configuration Interaction (QSCI) method, which is called in different ways in the literature, such as
- Time-Evolved QSCI (TE-QSCI) [arXiv:2412.13839](https://arxiv.org/abs/2412.13839)
- samplebased Krylov quantum diagonalization (SKQD) [arXiv:2501.09702](https://arxiv.org/abs/2501.09702)
- Hamiltonian simulation-based QSCI (HSB-QSCI) [arXiv:2412.07218](https://arxiv.org/abs/2412.07218)

In these methods, one starts with a simple reference state, such as Hartree-Fock state, and then time-evolution is performed to generate states including various excitations.
Then, the sampled bitstrings are used to construct a subspace of the configurations, which is then diagonalized in a classical computer.

In the following, we will use a noise-free Hamiltonian simulation for simplicity.
However, in practice, one should consider a way to recover e.g. Total-M, parity, and particle number for the sampled bitstrings.

Diagonalization within the sampled subspace can be done using [NuclearToolkit.jl](https://github.com/SotaYoshida/NuclearToolkit.jl) for smaller systems.

In [ ]:
fn = int_file_path+"ckpot.snt"
Z = N = 2
Hamil_Shellmodel, H_JW, proton_qubits, neutron_qubits = get_Hamiltonian(fn, Z, N)
n_qubits = Hamil_Shellmodel.n_qubits
mapper = JordanWignerMapper()

qc_init = QuantumCircuit(n_qubits)
qc_init.x(0); qc_init.x(1)
qc_init.x(6); qc_init.x(7)

In [ ]:
sampler = SamplerV2()
nshot = 1024
delta_t = 0.5

sampled_bitstrings = { }
for trange in range(5):
    t = delta_t * trange
    Unitary = PauliEvolutionGate(H_JW, t, synthesis=SuzukiTrotter(order=1,reps=1))
    qc_TE = qc_init.copy()
    if trange > 0:
        qc_TE.append(Unitary, range(n_qubits))
    tqc = qc_TE.decompose(reps=5)
    tqc.measure_all()
    
    job = sampler.run([tqc], shots=nshot)
    results = job.result()
    counts = results[0].data.meas.get_counts()      

    for (key, value) in counts.items():
        if key in sampled_bitstrings:
            sampled_bitstrings[key] += value
        else:
            sampled_bitstrings[key] = value
    print(f"trange={trange}, counts:", counts)

ordered_sampled_bitstrings = sorted(sampled_bitstrings.items(), key=lambda x:x[1], reverse=True)
print(ordered_sampled_bitstrings)
print("# of sampled configs.", len(ordered_sampled_bitstrings))


## (4) Solving valence-2n with Observable DMD

Under ckpot.snt, the low-lying spectra for valence-2n system, ${}^6$ He is the following:

| J  | Energy (MeV) |
|----|---------------|
| 0  | -3.910 | 
| 2  | 0.632  |
| 2  | 4.117  |
| 1  | 4.282  |
| 0  | 7.921   |

States with all possible Total-J are described within $M=0$ configurations.
In quantum computing, however, the overlap between initial state and exact wave function
would be a key factor in determining the success of the computation.

Firstly, we start from a lowest-filling configuration, $|j_z|=3/2$ pair in $p_{3/2}$ orbit.

In [13]:
Z = 0; N = 2
fn_snt = int_file_path+"ckpot.snt"
hamil, H_mapped, proton_qubits, neutron_qubits = get_Hamiltonian(fn_snt, Z, N)

# of H_m terms, 1b: 12, 2b pp: 43, nn: 43, pn: 276
Removing redundant terms... len= 3376


3376it [00:00, 214502.75it/s]


In [15]:
method = 0
if method == 0: # Statevector noise-free simulation
    using_statevector = True
    using_noisy_simulator = False
    sampler = None
    backend = None
elif method == 1: # noise-free shot-based simulation
    using_statevector = False
    using_noisy_simulator = False
    sampler = SamplerV2()
    backend = None
elif method == 2: # noisy shot measurement with FakeBackends
    using_statevector = False
    using_noisy_simulator = True
    backend = FakeTorino()
    sampler = AerSimulator.from_backend(backend)

delta_t = 0.01234
trotter_steps = 20
max_iterations = 15

Norb = n_qubit = hamil.n_qubits
proton_qubits = hamil.proton_qubits
neutron_qubits = hamil.neutron_qubits

dummy_params = [ ]
U_prep = nucl_ansatz(n_qubit, proton_qubits, neutron_qubits, Z, N, dummy_params, method="HF")
U_prep = QuantumCircuit(n_qubit)
U_prep.x([9, 10])


ancilla_qubits=[0]
target_qubits=list(range(1,Norb+1))

E0 = ODMD(U_prep, H_mapped, delta_t, max_iterations, trotter_steps, 
          sampler, backend, ancilla_qubits, target_qubits,
          num_shot=10**8,
          using_statevector=using_statevector, d=10)

  7%|▋         | 1/14 [00:05<01:17,  5.99s/it]

overlap   1: (0.9986600690689896-0.007850432108569394j)


 14%|█▍        | 2/14 [00:13<01:25,  7.10s/it]

overlap   2: (0.9946475918812294-0.015599465131949652j)


 21%|██▏       | 3/14 [00:18<01:07,  6.13s/it]

...

overlap  12: (0.8191585917768341-0.06724102013321631j)


 93%|█████████▎| 13/14 [01:13<00:06,  6.01s/it]

overlap  13: (0.7901402670076939-0.06818279986152814j)


100%|██████████| 14/14 [01:19<00:00,  5.69s/it]

overlap  14: (0.7595505672514704-0.06815880364620036j)
Max iteration:    15 trotter_steps:    20 delta_t:   0.01234000  tol for lambda = 1.00e-02
Check |AX - Y| 6.0654204549972995e-12
lam [0.99475262-0.09768831j 0.9966051 -0.0498876j  0.99790595-0.0055919j
 0.99871067+0.04849567j]
|lam| [0.99953779 0.99785294 0.99792162 0.99988741]
Ens: [7.9327182259136775, 4.053143351960069, 0.4540987926274012, -3.9319421015176794]
E0:   0.45409879 E closest to unit circle:  -3.93194210



Starting from a type reversal pair configuration in $p_{3/2}$ orbit,
J=0 states are rather well reproduced, but $J \neq 0$ states are not so well.

Next, we will investigate the effect of different initial states on the energy spectra
by taking $p_{3/2}$ and $p_{1/2}$ orbit configurations as examples.


In [ ]:
U_prep = nucl_ansatz(n_qubit, proton_qubits, neutron_qubits, Z, N, dummy_params, method="HF")
U_prep = QuantumCircuit(n_qubit)
U_prep.x([7, 9])

ancilla_qubits=[0]
target_qubits=list(range(1,Norb+1))

E0 = ODMD(U_prep, H_mapped, delta_t, max_iterations, trotter_steps, 
          sampler, backend,
          ancilla_qubits, target_qubits,
          using_statevector=using_statevector, d=10)

One can see that J=2 states (0.632&4.117 MeV) came out.

Finally, we will show the results with the $(p_{1/2})^2$ configuration:

In [ ]:
U_prep = nucl_ansatz(n_qubit, proton_qubits, neutron_qubits, Z, N, dummy_params, method="HF")
U_prep = QuantumCircuit(n_qubit)
U_prep.h(6)
U_prep.cx(6, 10)
U_prep.x(6)
U_prep.cx(7, 9)
U_prep.x(6)

ancilla_qubits=[0]
target_qubits=list(range(1,Norb+1))

E0 = ODMD(U_prep, H_mapped, delta_t, max_iterations, trotter_steps, 
          sampler, backend,
          ancilla_qubits, target_qubits,
          using_statevector=using_statevector, d=10,
          tol_lambda=1.e-2)